In [7]:
import pandas as pd
from openai import OpenAI

# Cliente (simula uso tipo LLaMA / GPT)
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Cargar datos
df = pd.read_excel(r"C:\Users\SD\Downloads\test_con_etiqueta.xlsx")

In [6]:
def clasificar_semantico(row):
    texto = (str(row.get("Título","")) + " " +
             str(row.get("Resumen","")) + " " +
             str(row.get("Artículos",""))).lower()

    score = 0
    razones = []

    if "víctima" in texto:
        score += 3
        razones.append("menciona víctimas")

    if "protección" in texto or "proteccion" in texto:
        score += 2
        razones.append("habla de protección")

    if "asistencia" in texto:
        score += 2
        razones.append("incluye asistencia")

    if "derechos" in texto:
        score += 1
        razones.append("menciona derechos")

    if "indemnización" in texto or "reparación" in texto:
        score += 2
        razones.append("incluye reparación")

    if score >= 3:
        return 1, "Contiene: " + ", ".join(razones)
    elif score > 0:
        return 0, "DUDOSO: menciones indirectas (" + ", ".join(razones) + ")"
    else:
        return 0, "No hay referencias a derechos de víctimas"


In [8]:
resultados = df.apply(clasificar_semantico, axis=1)

df["caso_ok"] = resultados.apply(lambda x: x[0])
df["Justificación"] = resultados.apply(lambda x: x[1])

In [14]:
df_final = df[["Tipo", "Número", "Título", "caso_ok", "Justificación"]]

df_final

,Tipo,Número,Título,caso_ok,Justificación
0,Ley,5209,EMERGENCIA PéBLICA,0,No hay referencias a derechos de víctimas
1,Ley,26004,ACUERDOS,1,"Contiene: menciona víctimas, incluye asistenci..."
2,Ley,25815,CODIGO PENAL. Modificación del mismo. Sustitúy...,1,"Contiene: menciona víctimas, incluye asistencia"
3,Ley,25632,CONVENCIONES,1,"Contiene: menciona víctimas, habla de protecci..."
4,Decreto Reglamentario,844,FONDO DE ASISTENCIA DIRECTA A VICTIMAS DE TRATA,1,"Contiene: menciona víctimas, incluye asistencia"
...,...,...,...,...,...
57,Ley,24417,PROTECCION CONTRA LA VIOLENCIA FAMILIAR,1,"Contiene: menciona víctimas, habla de protecci..."
58,Ley,6285,SUBSIDIO ESTATAL,0,No hay referencias a derechos de víctimas
59,Ley,27146,JUSTICIA,1,Contiene: menciona víctimas
60,Ley,25763,DERECHOS DEL NIÑO,1,"Contiene: menciona víctimas, habla de protecci..."


In [15]:
df_final.to_excel(r"C:\Users\SD\Downloads\resultado_clasificacion_oLlama.xlsx", index=False)

In [13]:
conteo = df["caso_ok"].value_counts()

print("Cantidad de 1:", conteo.get(1, 0))
print("Cantidad de 0:", conteo.get(0, 0))

Cantidad de 1: 34
Cantidad de 0: 28
